[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/raya-lucaria/ia_o26/blob/main/course/10_deep_learning/_assets/01_red_desde_cero.ipynb)

# Notebook 1 · Una red desde cero

Acompaña la [Clase 1 · Redes neuronales](https://raya-lucaria.github.io/ia_o26/deep-learning/redes-neuronales/). Solo usa numpy y
matplotlib, que Colab ya trae.

- Primero: **Entorno de ejecución → Ejecutar todas** (`Ctrl + F9`), o corre
  la celda «▶ Corre esta celda primero» y sigue de arriba abajo.
- Las celdas con funciones son el código de la página, copiado tal cual: no
  hace falta nada más.
- Al final hay tres **apuestas**: escribe tu predicción *antes* de correr.

In [ ]:
#@title ▶ Corre esta celda primero
import copy

import matplotlib.pyplot as plt
import numpy as np

plt.style.use("default")                     # fondo claro
plt.rcParams.update({"figure.facecolor": "white", "axes.grid": True,
                     "grid.alpha": 0.3, "font.size": 11})


def fmt(v):
    """Un número o un vector con 3 decimales: 0.546, [0.200, 0.000]."""
    v = np.round(np.asarray(v, dtype=float), 3) + 0.0
    if v.ndim == 0:
        return f"{v:.3f}"
    return "[" + ", ".join(f"{t:.3f}" for t in v) + "]"


def mostrar(nombre, M):
    """Una matriz con sus ejes: filas = destino i, columnas = origen j."""
    filas, columnas = M.shape
    print(f"{nombre}: {filas} × {columnas}  (filas = destino i, "
          "columnas = origen j)")
    print("       " + "".join(f"   j = {j}" for j in range(1, columnas + 1)))
    for i in range(filas):
        valores = np.round(M[i], 3) + 0.0
        print(f"i = {i + 1}  " + "".join(f"{t:8.3f}" for t in valores))


print("Listo: numpy", np.__version__)

## 1 · La red 2–3–2–1

La capa 0 es la entrada $\mathbf x$; las capas $l = 1, \dots, L$ tienen
parámetros. La capa $l$ tiene $d_l$ neuronas. Aquí $L = 3$ y
$d_0, d_1, d_2, d_3 = 2, 3, 2, 1$.

- $w^{(l)}_{ij}$ es el peso que va de la neurona $j$ de la capa $l-1$
  (**origen**) a la neurona $i$ de la capa $l$ (**destino**).
- $\mathbf W^{(l)} \in \mathbb R^{d_l \times d_{l-1}}$ junta esos pesos:
  filas = destino $i$, columnas = origen $j$.
- $\mathbf b^{(l)} \in \mathbb R^{d_l}$ es el sesgo de la capa $l$.

**Aviso.** Aquí $i$ y $j$ numeran **neuronas**. Las observaciones llevan el
índice $k$: $(\mathbf x_k, \mathbf y_k)$, con $k = 1, \dots, n$.

En el código, `red["W"][l]` es $\mathbf W^{(l)}$ y `red["W"][l][i-1, j-1]` es
$w^{(l)}_{ij}$ (Python cuenta desde 0). La posición 0 de cada lista vale
`None`: la entrada no tiene parámetros.

In [ ]:
def ejemplo():
    """La red 2–3–2–1 del hilo: ocultas ReLU, salida identidad, y (x, y).

    Elegida para que la capa 1 tenga una neurona inactiva (z^(1)_2 < 0,
    g' = 0) y las otras dos activas, y la capa 2 ambas activas (g' = 1).
    """
    red = {
        "W": [None,
              np.array([[0.5, -0.2],
                        [-0.4, 0.1],
                        [0.3, 0.2]]),            # W^(1): 3 × 2
              np.array([[0.4, -0.3, 0.5],
                        [0.2, 0.6, -0.1]]),      # W^(2): 2 × 3
              np.array([[0.7, -0.5]])],          # W^(3): 1 × 2
        "b": [None,
              np.array([0.1, 0.0, -0.1]),        # b^(1)
              np.array([0.1, 0.2]),              # b^(2)
              np.array([0.3])],                  # b^(3)
        "g": [None, "relu", "relu", "identidad"],
    }
    x = np.array([1.0, 2.0])
    y = np.array([1.0])
    return red, x, y


ETA_EJEMPLO = 0.1      # η del paso de la página 5
MU_EJEMPLO = 0.01      # μ del paso con decaimiento de pesos


def capas(red):
    """L: el número de capas con parámetros."""
    return len(red["W"]) - 1

In [ ]:
red, x, y = ejemplo()
L = capas(red)
anchos = [len(x)] + [len(b) for b in red["b"][1:]]
print("anchos d_0, ..., d_L:", anchos, "  L =", L, "\n")
for l in range(1, L + 1):
    mostrar(f"W^({l})", red["W"][l])
    print(f"b^({l}) = {fmt(red['b'][l])}   g^({l}) = {red['g'][l]}\n")

Cuenta los parámetros: $\mathbf W^{(l)}$ tiene $d_l \cdot d_{l-1}$ entradas y
$\mathbf b^{(l)}$ tiene $d_l$. Todos juntos son $\theta$.

In [ ]:
for l in range(1, L + 1):
    print(f"capa {l}: {red['W'][l].size} pesos + {red['b'][l].size} sesgos")
print("total de parámetros en θ:",
      sum(red["W"][l].size + red["b"][l].size for l in range(1, L + 1)))

## 2 · Hacia adelante

Cada capa hace dos cosas:

$$\mathbf z^{(l)} = \mathbf W^{(l)} \mathbf a^{(l-1)} + \mathbf b^{(l)},
\qquad \mathbf a^{(l)} = g^{(l)}\bigl(\mathbf z^{(l)}\bigr),
\qquad \mathbf a^{(0)} = \mathbf x .$$

$\mathbf z^{(l)}$ es la preactivación y $\mathbf a^{(l)}$ la activación. La
red es la composición $f_\theta = f^{(L)} \circ \cdots \circ f^{(1)}$, con
$f^{(l)}(\mathbf u) = g^{(l)}(\mathbf W^{(l)} \mathbf u + \mathbf b^{(l)})$, y
la predicción es $\widehat{\mathbf y} = f_\theta(\mathbf x) = \mathbf a^{(L)}$.

**Aviso.** La activación se llama $g^{(l)}$, no $\sigma$: en la unidad 9,
$\sigma$ es la sigmoide. Aquí la sigmoide es una $g$ posible; las ocultas
usan ReLU, $g(z) = \max(0, z)$, y la salida usa la identidad.

In [ ]:
def relu(z):
    return np.maximum(z, 0.0)


def relu_prima(z):
    return (z > 0).astype(float)


def sigmoide(z):
    return 1.0 / (1.0 + np.exp(-z))


def sigmoide_prima(z):
    s = sigmoide(z)
    return s * (1.0 - s)


def identidad(z):
    return z


def identidad_prima(z):
    return np.ones_like(z)


ACTIVACIONES = {
    "relu": (relu, relu_prima),
    "sigmoide": (sigmoide, sigmoide_prima),
    "identidad": (identidad, identidad_prima),
}

La pérdida de **una** observación es $\ell(\widehat{\mathbf y}, \mathbf y) =
\sum (\widehat y - y)^2$, sin $\tfrac12$ (igual que el MSE de la unidad 9).

In [ ]:
def adelante(red, x):
    """Pase hacia adelante. x es un vector, o una matriz d_0 × m (columnas =
    observaciones). Devuelve las listas z (z[0] = None) y a (a[0] = x) y ŷ."""
    z, a = [None], [x]
    for l in range(1, capas(red) + 1):
        b = red["b"][l] if x.ndim == 1 else red["b"][l][:, None]
        g, _ = ACTIVACIONES[red["g"][l]]
        z.append(red["W"][l] @ a[l - 1] + b)
        a.append(g(z[l]))
    return z, a, a[-1]


def perdida(y_gorro, y):
    """ℓ(ŷ, y) = Σ (ŷ − y)² de UNA observación."""
    return float(np.sum((y_gorro - y) ** 2))

In [ ]:
z, a, y_gorro = adelante(red, x)
print(f"a^(0) = x = {fmt(x)}")
for l in range(1, L + 1):
    print(f"capa {l}: z^({l}) = {fmt(z[l])}  →  a^({l}) = {fmt(a[l])}")
print(f"\nŷ = {fmt(y_gorro[0])}   y = {fmt(y[0])}")
print(f"ℓ(ŷ, y) = (ŷ − y)² = {fmt(perdida(y_gorro, y))}")

**Mira.** En la capa 1, $z^{(1)}_2$ es negativa: ReLU la vuelve 0 y esa
neurona no pasa nada hacia adelante. En la sección 4 vas a ver que tampoco
recibe nada hacia atrás.

## 3 · La pérdida de un conjunto y el lagrangeano

Con $n$ observaciones, la pérdida es el promedio, como en la unidad 9:

$$L(\theta) = \frac1n \sum_{k=1}^{n}
\ell\bigl(f_\theta(\mathbf x_k), \mathbf y_k\bigr) = \mathrm{MSE},
\qquad \mathrm{RMSE} = \sqrt{\mathrm{MSE}} .$$

Para que la red generalice, se limita su capacidad: minimizar $L(\theta)$
sujeto a $\Omega(\theta) \le d$, con $\Omega(\theta) = \sum_l
\lVert \mathbf W^{(l)} \rVert_F^2$ (ridge; los sesgos no se penalizan). El
lagrangeano junta las dos cosas, con $\mu \ge 0$:

$$\mathcal L(\theta, \mu) = L(\theta) + \mu\,\bigl(\Omega(\theta) - d\bigr).$$

En el código, `X` y `Y` llevan **una columna por observación** $k$. Un
vector suelto cuenta como una sola columna. Con $\mu$ fijo, $d$ no cambia
dónde está el mínimo; por eso el código usa $d = 0$ si no se le da otro.

In [ ]:
def _columnas(v):
    """Un vector suelto es una sola observación: una columna."""
    return v[:, None] if v.ndim == 1 else v


def perdida_promedio(red, X, Y):
    """L(θ) = (1/n) Σ_k ℓ(f_θ(x_k), y_k); para regresión es el MSE."""
    X, Y = _columnas(X), _columnas(Y)
    _, _, Y_gorro = adelante(red, X)
    return float(np.mean(np.sum((Y_gorro - Y) ** 2, axis=0)))


def rmse(red, X, Y):
    """RMSE = √MSE."""
    return float(np.sqrt(perdida_promedio(red, X, Y)))


def omega(red):
    """Ω(θ) = Σ_l ‖W^(l)‖_F²: suma de los cuadrados de todos los pesos."""
    return float(sum(np.sum(W ** 2) for W in red["W"][1:]))


def lagrangeano(red, X, Y, mu=0.0, d=0.0):
    """𝓛(θ, μ) = L(θ) + μ (Ω(θ) − d)."""
    return perdida_promedio(red, X, Y) + mu * (omega(red) - d)

In [ ]:
print(f"Ω(θ) = Σ_l ‖W^(l)‖² = {fmt(omega(red))}")
print(f"L(θ) con la única observación = {fmt(perdida_promedio(red, x, y))}")
print(f"RMSE = √L = {fmt(rmse(red, x, y))}")
for mu in (0.0, MU_EJEMPLO):
    print(f"𝓛(θ, μ = {mu}) = {fmt(lagrangeano(red, x, y, mu))}")

## 4 · Retropropagación

El error retropropagado de la capa $l$ es $\boldsymbol\delta^{(l)} :=
\nabla_{\mathbf z^{(l)}} \ell$, de **una** observación. Se calcula de la
salida hacia la entrada:

$$\boldsymbol\delta^{(L)} = \nabla_{\mathbf a^{(L)}} \ell \odot
g^{(L)\prime}\bigl(\mathbf z^{(L)}\bigr), \qquad
\boldsymbol\delta^{(l)} = \bigl(\mathbf W^{(l+1)\mathsf T}
\boldsymbol\delta^{(l+1)}\bigr) \odot g^{(l)\prime}\bigl(\mathbf z^{(l)}\bigr).$$

Con $\ell = (\widehat y - y)^2$, $\nabla_{\mathbf a^{(L)}} \ell =
2(\widehat{\mathbf y} - \mathbf y)$. Y los gradientes salen de $\boldsymbol
\delta$:

$$\nabla_{\mathbf W^{(l)}} \ell = \boldsymbol\delta^{(l)}
\mathbf a^{(l-1)\mathsf T}, \qquad \nabla_{\mathbf b^{(l)}} \ell =
\boldsymbol\delta^{(l)}, \qquad \frac{\partial \ell}{\partial w^{(l)}_{ij}}
= \delta^{(l)}_i\, a^{(l-1)}_j .$$

Error del destino por activación del origen. En ReLU se toma
$g'(0) = 0$.

In [ ]:
def atras(red, x, y):
    """Retropropagación de una observación (x, y).

    δ^(L) = 2(ŷ − y) ⊙ g^(L)'(z^(L));
    δ^(l) = (W^(l+1)ᵀ δ^(l+1)) ⊙ g^(l)'(z^(l));
    ∇_{W^(l)} ℓ = δ^(l) a^(l-1)ᵀ;  ∇_{b^(l)} ℓ = δ^(l).
    """
    z, a, y_gorro = adelante(red, x)
    L = capas(red)
    delta = [None] * (L + 1)
    grad_W, grad_b = [None] * (L + 1), [None] * (L + 1)
    for l in range(L, 0, -1):
        _, g_prima = ACTIVACIONES[red["g"][l]]
        if l == L:
            arriba = 2 * (y_gorro - y)                 # ∇_{a^(L)} ℓ
        else:
            arriba = red["W"][l + 1].T @ delta[l + 1]
        delta[l] = arriba * g_prima(z[l])
        grad_W[l] = np.outer(delta[l], a[l - 1])
        grad_b[l] = delta[l].copy()
    return delta, grad_W, grad_b

In [ ]:
delta, grad_W, grad_b = atras(red, x, y)
for l in range(L, 0, -1):
    print(f"δ^({l}) = {fmt(delta[l])}")
    mostrar(f"∇_{{W^({l})}} ℓ = δ^({l}) a^({l - 1})ᵀ", grad_W[l])
    print(f"∇_{{b^({l})}} ℓ = δ^({l}) = {fmt(grad_b[l])}\n")

**Mira.** $\delta^{(1)}_2 = 0$: la neurona apagada tiene $g' = 0$, así que
toda la fila 2 de $\nabla_{\mathbf W^{(1)}} \ell$ es cero. Y cada entrada es
«error del destino × activación del origen»:

In [ ]:
i, j = 1, 2                          # el peso w^(1)_12 (índices desde 1)
print(f"δ^(1)_{i} · a^(0)_{j} = {fmt(delta[1][i - 1] * x[j - 1])}")
print(f"∂ℓ/∂w^(1)_{i}{j}      = {fmt(grad_W[1][i - 1, j - 1])}")

**¿Está bien?** Se comprueba sin derivar: se mueve un solo parámetro un
poquito, $h$, hacia arriba y hacia abajo, y se mide el cambio de la pérdida:

$$\frac{\mathcal L(\theta + h\,\mathbf e) - \mathcal L(\theta - h\,\mathbf e)}
{2h} .$$

`gradiente` es la retropropagación de un **lote** (columnas = observaciones):
promedia los gradientes de $\ell$ y suma $2\mu\mathbf W^{(l)}$ a cada
$\mathbf W^{(l)}$. Con una sola observación y $\mu = 0$ da lo mismo que
`atras`. El error relativo compara los dos:
$\lVert a - n \rVert / (\lVert a \rVert + \lVert n \rVert)$.

In [ ]:
def gradiente(red, X, Y, mu=0.0):
    """∇_θ 𝓛 de un lote (columnas = observaciones): los gradientes de ℓ se
    promedian sobre el lote y a cada W^(l) se le suma 2 μ W^(l)."""
    X, Y = _columnas(X), _columnas(Y)
    z, a, Y_gorro = adelante(red, X)
    L, m = capas(red), X.shape[1]
    grad_W, grad_b = [None] * (L + 1), [None] * (L + 1)
    for l in range(L, 0, -1):
        _, g_prima = ACTIVACIONES[red["g"][l]]
        if l == L:
            arriba = 2 * (Y_gorro - Y)
        else:
            arriba = red["W"][l + 1].T @ delta
        delta = arriba * g_prima(z[l])                   # d_l × m
        grad_W[l] = delta @ a[l - 1].T / m + 2 * mu * red["W"][l]
        grad_b[l] = delta.mean(axis=1)
    return grad_W, grad_b

In [ ]:
def gradiente_numerico(red, X, Y, mu=0.0, h=1e-6):
    """∇_θ 𝓛 por diferencias finitas centradas, una entrada a la vez."""
    grad = {"W": [None], "b": [None]}
    for nombre in ("W", "b"):
        for l in range(1, capas(red) + 1):
            G = np.zeros_like(red[nombre][l])
            for idx in np.ndindex(G.shape):
                mas, menos = copy.deepcopy(red), copy.deepcopy(red)
                mas[nombre][l][idx] += h
                menos[nombre][l][idx] -= h
                G[idx] = (lagrangeano(mas, X, Y, mu)
                          - lagrangeano(menos, X, Y, mu)) / (2 * h)
            grad[nombre].append(G)
    return grad["W"], grad["b"]


def comprobar_gradiente(red, X, Y, mu=0.0, h=1e-6):
    """Error relativo máximo, por bloque W^(l) y b^(l), entre la
    retropropagación y las diferencias finitas: ‖a − n‖ / (‖a‖ + ‖n‖)."""
    analitico = gradiente(red, X, Y, mu)
    numerico = gradiente_numerico(red, X, Y, mu, h)
    peor = 0.0
    for A_bloques, N_bloques in zip(analitico, numerico):
        for A, N in zip(A_bloques[1:], N_bloques[1:]):
            escala = np.linalg.norm(A) + np.linalg.norm(N)
            if escala > 0:
                peor = max(peor, np.linalg.norm(A - N) / escala)
    return float(peor)

In [ ]:
grad_W_num, grad_b_num = gradiente_numerico(red, x, y)
mostrar("∇_{W^(1)} ℓ por diferencias finitas", grad_W_num[1])
error = comprobar_gradiente(red, x, y)
error_mu = comprobar_gradiente(red, x, y, MU_EJEMPLO)
print(f"\nerror relativo máximo, μ = 0:    {error:.1e}")
print(f"error relativo máximo, μ = {MU_EJEMPLO}: {error_mu:.1e}")
print("¿coinciden?", error < 1e-6 and error_mu < 1e-6)

## 5 · Un paso de descenso

Se mueve $\theta$ contra el gradiente, con una tasa de aprendizaje
$\eta > 0$ (una letra nueva: en la unidad 9 no aparece):

$$\theta \leftarrow \theta - \eta\, \nabla_\theta \mathcal L, \qquad
\mathbf W^{(l)} \leftarrow \mathbf W^{(l)} - \eta\bigl(\nabla_{\mathbf
W^{(l)}} L + 2\mu \mathbf W^{(l)}\bigr).$$

El término $2\mu\mathbf W^{(l)}$ encoge cada peso un poco en cada paso: es
el **decaimiento de pesos**. Aquí $\eta = 0.1$ y $\mu = 0.01$.

In [ ]:
def paso(red, X, Y, eta, mu=0.0):
    """Un paso de descenso: W ← W − η(∇_W L + 2μW), b ← b − η ∇_b L.
    Devuelve una red nueva; no toca la que recibe."""
    grad_W, grad_b = gradiente(red, X, Y, mu)
    nueva = {"W": [None], "b": [None], "g": list(red["g"])}
    for l in range(1, capas(red) + 1):
        nueva["W"].append(red["W"][l] - eta * grad_W[l])
        nueva["b"].append(red["b"][l] - eta * grad_b[l])
    return nueva

In [ ]:
nueva = paso(red, x, y, ETA_EJEMPLO)
y_nueva = adelante(nueva, x)[2]
print(f"η = {ETA_EJEMPLO}")
print(f"antes:   ŷ = {fmt(y_gorro[0])}   ℓ = {fmt(perdida(y_gorro, y))}"
      f"   Ω = {fmt(omega(red))}")
print(f"después: ŷ = {fmt(y_nueva[0])}   ℓ = {fmt(perdida(y_nueva, y))}"
      f"   Ω = {fmt(omega(nueva))}")
nueva_mu = paso(red, x, y, ETA_EJEMPLO, MU_EJEMPLO)
y_mu = adelante(nueva_mu, x)[2]
print(f"con μ = {MU_EJEMPLO}: ŷ = {fmt(y_mu[0])}   ℓ = {fmt(perdida(y_mu, y))}"
      f"   Ω = {fmt(omega(nueva_mu))}")

Un paso basta para que $\ell$ baje mucho. Con $\mu > 0$, $\Omega$ crece
menos: el decaimiento frena a los pesos. Ahora cinco pasos seguidos:

In [ ]:
red_t = red
for t in range(6):
    print(f"paso {t}: ℓ = {fmt(perdida(adelante(red_t, x)[2], y))}")
    red_t = paso(red_t, x, y, ETA_EJEMPLO)

## 6 · Entrenar sin μ y con μ

Ahora una red de verdad, $2–32–32–1$ con ReLU, sobre datos sintéticos:
$y = \sin(\pi x_1)\cos(\pi x_2 / 2)$ más ruido normal de desviación
$0.3$, con $80$ observaciones de entrenamiento y $200$ de
validación.

- Pesos iniciales normales escalados por $\sqrt{2 / d_{l-1}}$; sesgos en 0.
- **Minibatch**: en cada época se barajan las observaciones y se da un paso
  por lote de $20$ ($4$ pasos por época), promediando los
  gradientes del lote.
- $\eta = 0.03$ y $1500$ épocas, sin $\mu$ y con $\mu = 0.003$: los
  mismos pesos iniciales y el mismo barajado en las dos corridas.

In [ ]:
SEMILLAS = {"datos": 4, "pesos": 3, "lotes": 103}

ANCHOS_ENTRENAMIENTO = (2, 32, 32, 1)
ACTIVACIONES_ENTRENAMIENTO = ("relu", "relu", "identidad")
N_ENTRENAMIENTO = 80
N_VALIDACION = 200
RUIDO = 0.3            # desviación estándar del ruido normal
ETA_ENTRENAMIENTO = 0.03
LOTE = 20              # 80 / 20 = 4 pasos por época
EPOCAS = 1500
MU_ENTRENAMIENTO = 0.003


def funcion_verdadera(X):
    """f(x₁, x₂) = sin(π x₁) · cos(π x₂ / 2), la curva que los datos siguen."""
    return np.sin(np.pi * X[0]) * np.cos(np.pi * X[1] / 2)


def datos_ejemplo(semilla=SEMILLAS["datos"]):
    """x uniforme en [−1, 1]², y = f(x) + ruido normal de desviación RUIDO.
    Devuelve X, Y (entrenamiento) y X_val, Y_val (validación), en columnas."""
    rng = np.random.default_rng(semilla)
    X = rng.uniform(-1, 1, (2, N_ENTRENAMIENTO))
    X_val = rng.uniform(-1, 1, (2, N_VALIDACION))
    Y = funcion_verdadera(X) + RUIDO * rng.standard_normal(N_ENTRENAMIENTO)
    Y_val = (funcion_verdadera(X_val)
             + RUIDO * rng.standard_normal(N_VALIDACION))
    return X, Y[None, :], X_val, Y_val[None, :]

In [ ]:
X, Y, X_val, Y_val = datos_ejemplo()
print("X:", X.shape, "  Y:", Y.shape, "  (columnas = observaciones k)")
print("X_val:", X_val.shape, "  Y_val:", Y_val.shape)

In [ ]:
def red_inicial(anchos, activaciones, semilla=SEMILLAS["pesos"]):
    """Pesos normales escalados por √(2 / d_{l-1}) (He); sesgos en cero."""
    rng = np.random.default_rng(semilla)
    red = {"W": [None], "b": [None], "g": [None, *activaciones]}
    for l in range(1, len(anchos)):
        escala = np.sqrt(2 / anchos[l - 1])
        red["W"].append(rng.standard_normal((anchos[l], anchos[l - 1]))
                        * escala)
        red["b"].append(np.zeros(anchos[l]))
    return red


def entrenar(red, X, Y, X_val, Y_val, eta, mu, epocas, lote,
             semilla=SEMILLAS["lotes"]):
    """Descenso por minibatch. En cada época se barajan las observaciones y
    se da un paso por lote. Historia: RMSE de entrenamiento y de validación
    antes de entrenar (época 0) y al final de cada época."""
    rng = np.random.default_rng(semilla)
    historia = {"rmse_entrenamiento": [rmse(red, X, Y)],
                "rmse_validacion": [rmse(red, X_val, Y_val)]}
    for _ in range(epocas):
        orden = rng.permutation(X.shape[1])
        for inicio in range(0, len(orden), lote):
            k = orden[inicio:inicio + lote]
            red = paso(red, X[:, k], Y[:, k], eta, mu)
        historia["rmse_entrenamiento"].append(rmse(red, X, Y))
        historia["rmse_validacion"].append(rmse(red, X_val, Y_val))
    return red, historia

In [ ]:
def entrenar_ejemplo(mu=0.0, epocas=EPOCAS):
    """El entrenamiento de la página 5: red 2–32–32–1 ReLU sobre los datos de
    datos_ejemplo(), mismos pesos iniciales y mismo barajado para todo μ.
    Devuelve la historia {rmse_entrenamiento, rmse_validacion}, cada una con
    epocas + 1 valores (el primero es antes de entrenar)."""
    X, Y, X_val, Y_val = datos_ejemplo()
    red = red_inicial(ANCHOS_ENTRENAMIENTO, ACTIVACIONES_ENTRENAMIENTO)
    _, historia = entrenar(red, X, Y, X_val, Y_val, ETA_ENTRENAMIENTO, mu,
                           epocas, LOTE)
    return historia

In [ ]:
sin_mu = entrenar_ejemplo(0.0)                     # unos segundos
con_mu = entrenar_ejemplo(MU_ENTRENAMIENTO)
for nombre, h in (("sin μ", sin_mu), (f"μ = {MU_ENTRENAMIENTO}", con_mu)):
    ent, val = h["rmse_entrenamiento"], np.array(h["rmse_validacion"])
    print(f"{nombre}: RMSE final ent. = {fmt(ent[-1])}, "
          f"val. = {fmt(val[-1])}; mejor val. = {fmt(val.min())} "
          f"en la época {val.argmin()}")

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
titulos = ("Sin μ", f"Con μ = {MU_ENTRENAMIENTO}")
for eje, h, titulo in zip(ejes, (sin_mu, con_mu), titulos):
    eje.plot(h["rmse_entrenamiento"], label="entrenamiento")
    eje.plot(h["rmse_validacion"], label="validación")
    eje.axhline(RUIDO, color="gray", ls="--", label="ruido")
    eje.set(title=titulo, xlabel="época", ylabel="RMSE", ylim=(0, 0.9))
    eje.legend()
fig.suptitle("RMSE por época, red " + "–".join(map(str, ANCHOS_ENTRENAMIENTO)))
plt.show()

**Lee las curvas.** Sin $\mu$, la RMSE de entrenamiento sigue bajando, pero
la de validación toca fondo y vuelve a subir: la red empieza a memorizar el
ruido. Con $\mu$, el entrenamiento baja menos y la validación termina más
abajo. Ninguna puede bajar mucho del ruido (la línea punteada) en
validación: ese error no se puede aprender.

## 7 · Apuesta y luego corre

Tres cambios de un número. Para cada uno: lee la pregunta, **escribe tu
apuesta** y solo después corre la celda. Las tres usan esta función, que
entrena con los mismos datos y el mismo arranque de la sección 6.

In [ ]:
def apuesta(eta=ETA_ENTRENAMIENTO, mu=0.0, anchos=ANCHOS_ENTRENAMIENTO,
            epocas=EPOCAS):
    red_0 = red_inicial(anchos, ACTIVACIONES_ENTRENAMIENTO)
    red_f, h = entrenar(red_0, X, Y, X_val, Y_val, eta, mu, epocas, LOTE)
    print(f"η = {eta}, μ = {mu}, red {'–'.join(map(str, anchos))}: "
          f"RMSE final ent. = {fmt(h['rmse_entrenamiento'][-1])}, "
          f"val. = {fmt(h['rmse_validacion'][-1])}")
    return red_f, h

**Apuesta 1 · η.** Con $\eta = 0.03$ la red aprende.
¿Qué pasa con $\eta = 0.5$?

- (a) aprende más rápido y termina mejor;
- (b) termina igual, pero llega antes;
- (c) se descompone y deja de aprender.

In [ ]:
red_eta, h_eta = apuesta(eta=0.5, epocas=300)
for t in (0, 1, 2, 10, 300):
    print(f"época {t:>3}:  η = {ETA_ENTRENAMIENTO} → "
          f"{fmt(sin_mu['rmse_entrenamiento'][t])}   "
          f"η = 0.5 → {fmt(h_eta['rmse_entrenamiento'][t])}")
y_eta = adelante(red_eta, X)[2]
print(f"\nŷ va de {fmt(y_eta.min())} a {fmt(y_eta.max())}; "
      f"desviación estándar de y: {fmt(Y.std())}")

<details><summary>Respuesta</summary>

**(c).** En la época 1 la RMSE salta a más de mil: el paso es tan largo
que se pasa del mínimo. Después, todas las $z$ de las ocultas quedan
negativas y ReLU las apaga ($g' = 0$): las $\boldsymbol\delta$ de las
ocultas valen cero y solo se mueve el sesgo de la salida. La red predice la
misma $\widehat y$ para todo, y su RMSE es la desviación estándar de $y$ (lo
mejor que hace una constante). Una $\eta$ grande no es «aprender más
rápido».

</details>

**Apuesta 2 · μ.** Con $\mu = 0.003$ la validación
terminó mejor que sin $\mu$. Con $\mu$ diez veces mayor,
$\mu = 0.03$, ¿la RMSE final de validación baja todavía más, o
sube? ¿Y la de entrenamiento?

In [ ]:
_, h_mu = apuesta(mu=0.03)
for nombre, h in (("μ = 0", sin_mu), (f"μ = {MU_ENTRENAMIENTO}", con_mu),
                  ("μ = 0.03", h_mu)):
    print(f"{nombre:>9}: ent. = {fmt(h['rmse_entrenamiento'][-1])}, "
          f"val. = {fmt(h['rmse_validacion'][-1])}")

<details><summary>Respuesta</summary>

**Suben las dos.** Con $\mu$ grande, $\Omega$ pesa más que $L$ en
$\mathcal L$: los pesos quedan tan chicos que la red ya no alcanza a seguir
la curva. Antes sobraba capacidad; ahora falta. Por eso $\mu$ se elige con
validación, como en la unidad 9: ni cero ni lo más grande posible.

</details>

**Apuesta 3 · el ancho.** Sin $\mu$, la red $2–32–32–1$ terminó con mucha
distancia entre entrenamiento y validación. Con solo 4 neuronas por capa
oculta, $(2–4–4–1)$ y sin $\mu$, ¿esa distancia crece o se
achica? ¿Cuál termina mejor en validación?

In [ ]:
_, h_ancho = apuesta(anchos=(2, 4, 4, 1))
for nombre, h in (("2–32–32–1", sin_mu), ("2–4–4–1", h_ancho)):
    ent, val = h["rmse_entrenamiento"][-1], h["rmse_validacion"][-1]
    print(f"{nombre:>9}: ent. = {fmt(ent)}, val. = {fmt(val)}, "
          f"distancia = {fmt(val - ent)}")

<details><summary>Respuesta</summary>

**Se achica.** La red angosta tiene muchos menos parámetros: no le alcanza
para memorizar el ruido, así que su RMSE de entrenamiento sube y la de
validación baja. Quitar neuronas es otra forma de limitar la capacidad,
igual que $\mu$: las dos acercan las curvas.

</details>

**Para cerrar.** Todo el entrenamiento es el mismo ciclo: hacia adelante
($\mathbf z$, $\mathbf a$, $\widehat{\mathbf y}$), la pérdida, hacia atrás
($\boldsymbol\delta$ y los gradientes) y un paso de descenso. Lo demás
(η, μ, el ancho) decide qué tan bien generaliza.